# Group 14 – Federal Government Spending: Where Does the Money Go?
**Course:** SEN 381 Data Science Projects  |  **Milestone 1:** Data acquired and documented

**Group members:** 
1. Dzarma Rebecca
2. Abibang Gift Ohihe
3. Chidi Kamsy Enyinna
4. Mitchell Olaide Bamidele
5. Obi Joe Osarr
6. Amos Alvin Okuwatamilore

## 1. Problem statement
The Federal Government of Nigeria publishes payments made by its Ministries, Departments and Agencies (MDAs) through the Open Treasury Portal. These records are large, spread over many daily files, and hard to read as a whole. This project combines the daily payment reports into one table and asks: **where does federal money go, who receives it, and when?** The aim is to describe spending by MDA, beneficiary and month, find unusual payments and seasonal spikes, and build a simple model that predicts whether an MDA's monthly spend (or a single payment) will be large.

## 2. Research questions
1. Which MDAs and beneficiary accounts receive the largest share of total spending?
2. How does spending change month to month, and is there an end-of-year spike?
3. Which individual payments are the largest, and how many payments account for most of the money?
4. Which payments are regular and predictable (for example payroll through IPPIS) and which are irregular?
5. Can a model trained on earlier months predict whether a later month's spend (or a single payment) is "large" (Yes/No)?

## 3. Data source and documentation
| Item | Detail |
|---|---|
| Source | Open Treasury Portal, Office of the Accountant-General of the Federation, daily payment reports |
| Link | https://opentreasury.gov.ng (Daily Payment Report – FGN section) |
| Downloaded on | 13-11-2026|
| Downloaded by | Obi Joe Osarr |
| Coverage in the file we hold | 2025-02-28 to 2025-12-29 (see section 5) |
| Raw file | `Group_14_-_Federal_Government_Yearly_Spending.csv` |
| Terms of use | No explicit terms of use or licence were found on the Open Treasury Portal at the time of download [date]. Data is published by the Federal Government under its Financial Transparency Policy for public transparency. We use it for non-commercial academic analysis, cite the portal as the source, and treat figures as published by each MDA, without independent verification. |    

**Known limits of the source**: the portal only publishes payments above a threshold, so small payments are missing, and some daily reports are missing or inaccessible. Treat totals as *published payments above the threshold*, not total government spending.

In [4]:
import pandas as pd

RAW = "Group 14 - Federal Government Yearly Spending.csv"

# First, look at the file exactly as it is
with open(RAW, encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(i, repr(line.strip()))
        if i == 10:
            break

0 'FEDERAL GOVERNMENT SPENDING RECORD,,,'
1 ',,,'
2 ',,,'
3 'DATE,MDA,Beneficiary,Amount (NGN)'
4 '2025-02-28,NIGERIAN NAVY,IPPIS TRANSACTION ACCOUNT,"?259,475,184.39"'
5 '2025-02-28,NIGERIAN NAVY,IPPIS TRANSACTION ACCOUNT,"?5,513,818,081.07"'
6 '2025-03-17,NIGERIAN NAVY,Naval Headquarters Maritime Operations Account,"?120,000,000.00"'
7 '2025-03-17,NIGERIAN NAVY,Naval Headquarters Maritime Operations Account,"?150,000,000.00"'
8 '2025-03-17,NIGERIAN NAVY,Naval Headquarters Maritime Operations Account,"?385,044,890.50"'
9 '2025-03-17,NIGERIAN NAVY,Naval Headquarters Maritime Operations Account,"?500,000,000.00"'
10 '2025-03-17,NIGERIAN NAVY,Naval Headquarters Maritime Operations Account,"?500,000,000.00"'


In [5]:
df = pd.read_csv(RAW, skiprows=3, dtype=str)#skiprows=3 is used to skip the first 3 rows of the CSV file which contains headers.
print(df.head(8).to_string())

         DATE            MDA                                     Beneficiary       Amount (NGN)
0  2025-02-28  NIGERIAN NAVY                       IPPIS TRANSACTION ACCOUNT    ?259,475,184.39
1  2025-02-28  NIGERIAN NAVY                       IPPIS TRANSACTION ACCOUNT  ?5,513,818,081.07
2  2025-03-17  NIGERIAN NAVY  Naval Headquarters Maritime Operations Account    ?120,000,000.00
3  2025-03-17  NIGERIAN NAVY  Naval Headquarters Maritime Operations Account    ?150,000,000.00
4  2025-03-17  NIGERIAN NAVY  Naval Headquarters Maritime Operations Account    ?385,044,890.50
5  2025-03-17  NIGERIAN NAVY  Naval Headquarters Maritime Operations Account    ?500,000,000.00
6  2025-03-17  NIGERIAN NAVY  Naval Headquarters Maritime Operations Account    ?500,000,000.00
7  2025-03-17  NIGERIAN NAVY  Naval Headquarters Maritime Operations Account  ?1,000,000,000.00


## 4. First look: shape, data types, missing values

In [3]:
print("Shape (rows, columns):", df.shape)
print()
print("Columns:", list(df.columns))
print()
print("Data types as loaded:")
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isna().sum())
print()
print("Exact duplicate rows:", df.duplicated().sum())

Shape (rows, columns): (50, 4)

Columns: ['DATE', 'MDA', 'Beneficiary', 'Amount (NGN)']

Data types as loaded:
DATE            str
MDA             str
Beneficiary     str
Amount (NGN)    str
dtype: object

Missing values per column:
DATE            0
MDA             0
Beneficiary     0
Amount (NGN)    0
dtype: int64

Exact duplicate rows: 2


In [4]:
print("Distinct values per column:")
print(df.nunique())
print()
print("Rows per MDA:")
print(df["MDA"].value_counts().to_string())
print()
print("Rows per beneficiary:")
print(df["Beneficiary"].value_counts().to_string())

Distinct values per column:
DATE            22
MDA              1
Beneficiary      3
Amount (NGN)    31
dtype: int64

Rows per MDA:
MDA
NIGERIAN NAVY    50

Rows per beneficiary:
Beneficiary
Naval Headquarters Maritime Operations Account    21
IPPIS TRANSACTION ACCOUNT                         15
CHIEF OF NAVAL STAFF PERSONNEL ACCOUNT            14
